# Phase 8 — Main iEnhancer-2L benchmark synthesis

**Goal:** produce traceable manuscript tables and figures from the audited main dataset and completed grouped out-of-fold experiments. This notebook performs **no training or model inference**. Run on Kaggle **CPU**, Internet **off**.

## Attach exactly these three inputs (ZIPs or Kaggle-extracted equivalents)

| Input | Required file in it | Purpose |
|---|---|---|
| `[1] enhancer_audit_v1_outputs.zip` | `stage1_candidate_manifest.csv`, `stage2_candidate_manifest.csv` | Audited sequences, roles, folds |
| `[2.1] enhancer_phase2_corrected_results.zip` | `all_oof_predictions.csv` | Corrected nine-model grouped OOF predictions |
| `3_enhancer_phase3_robustness.zip` | `phase3_model_robustness.csv` | Exact-GC diagnostics and conditional bootstrap |

Attach each as an **Input** to this Kaggle notebook. Use *Run All*; download `enhancer_phase8_main_synthesis.zip` from the notebook Output. **No `.pt` files, Phase 4–7 archives, GPU, or pretrained-model downloads are needed.**

Interpretation: five-fold grouped OOF comparisons on a sequence-similarity graph. The historical independent set is previously examined; these results are **not** an untouched independent test. MPRA Phase 6–7 findings belong in a separate external-validation section of the eventual paper.

In [ ]:
# 0 — Imports, output folder, and fixed source identities.
from pathlib import Path
import io, json, hashlib, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

WORK = Path('/kaggle/working/phase8_main_synthesis') if Path('/kaggle/working').exists() else Path.cwd()/'phase8_main_synthesis'
WORK.mkdir(parents=True, exist_ok=True)
EXPECTED_REPORT_SHA = {
 'audit_run_summary.json':'39ef44654b5576f4e71491f96e14e6816d0970a0a49ae3581d51cfb0f05a3263',
 'correction_report.json':'8296e5703c8e73765b42923be970b6752061639331c1318c6a3414767ed6914a',
 'phase3_run_report.json':'e6c91050f7b9beedde686d11484cbe5ca30fb462c7c748c2f68cea4119626049'
}
EXPECTED_MANIFEST = {
 'stage1_candidate_manifest.csv':'e295155300fafc34528bc1868c6fad6ec78f3912c5b07e034710e81c3b4ac4b3',
 'stage2_candidate_manifest.csv':'e25291665c8c42522b662855d3774a0e425e37f603c2ab1f22f4a2c241def952'
}
EXPECTED_CONFIG_HASH = '63e47ce62c02127cc6bf71762c1205d3bf473cc84085cae3f455ceffb7b1e5a3'
def sha(raw): return hashlib.sha256(raw).hexdigest()
def table(raw): return pd.read_csv(io.BytesIO(raw), dtype=str, keep_default_na=False)
print('Output folder:', WORK)

In [ ]:
# 1 — Locate each archive or an extracted folder by its distinctive report.
# Deduplicate identical copies, fail on two different versions. No filename assumptions.
class Source:
 def __init__(self, path, zipped): self.path, self.zipped = path, zipped
 def read(self, name):
  if self.zipped:
   with zipfile.ZipFile(self.path) as z:
    matches=[i.filename for i in z.infolist() if not i.is_dir() and i.filename.split('/')[-1]==name]
    if len(matches)!=1: raise RuntimeError(f'{self.path}: expected one {name}, found {matches}')
    return z.read(matches[0])
  paths=list(self.path.glob(name))
  if len(paths)!=1: raise RuntimeError(f'{self.path}: expected one {name}, found {paths}')
  return paths[0].read_bytes()
 def __str__(self): return str(self.path)

def locate(marker):
 roots=[Path('/kaggle/input')] if Path('/kaggle/input').exists() else [Path.cwd()]
 found=[]
 for root in roots:
  for p in root.rglob(marker):
   if p.is_file(): found.append(Source(p.parent,False))
  for p in root.rglob('*.zip'):
   if not p.is_file(): continue
   try:
    with zipfile.ZipFile(p) as z:
     if any(x.split('/')[-1]==marker for x in z.namelist()): found.append(Source(p,True))
   except zipfile.BadZipFile: continue
 versions={}
 for src in found:
  raw=src.read(marker); versions.setdefault(sha(raw),[]).append(src)
 if len(versions)!=1:
  raise RuntimeError(f'Expected one version containing {marker}; found '+str({k:[str(s) for s in v] for k,v in versions.items()}))
 # Prefer the actual ZIP when Kaggle offers both ZIP and extracted files.
 chosen=sorted(next(iter(versions.values())),key=lambda s:(not s.zipped,len(str(s.path))))[0]
 assert sha(chosen.read(marker))==EXPECTED_REPORT_SHA[marker], f'Unexpected source report: {marker} in {chosen}'
 print('Verified source:', marker, 'from', chosen)
 return chosen

P1=locate('audit_run_summary.json')
P2=locate('correction_report.json')
P3=locate('phase3_run_report.json')

def verify_members(src, report):
 hashes=report.get('output_sha256',report.get('outputs_sha256'))
 assert isinstance(hashes,dict)
 for name,digest in hashes.items():
  assert sha(src.read(name))==digest, f'Integrity mismatch: {name} in {src}'

r1=json.loads(P1.read('audit_run_summary.json'))
r2=json.loads(P2.read('correction_report.json'))
r3=json.loads(P3.read('phase3_run_report.json'))
verify_members(P1,r1); verify_members(P2,r2); verify_members(P3,r3)
assert r2['input_manifest_sha256']==r3['phase1_manifest_sha256']==EXPECTED_MANIFEST
assert r2['config_hash']==r3['phase2_config_hash']==EXPECTED_CONFIG_HASH
lock=json.loads(P2.read('input_lock.json'))
assert lock['manifest_sha256']==EXPECTED_MANIFEST and lock['config_hash']==EXPECTED_CONFIG_HASH
# Outer ZIP bytes can change when Kaggle repacks an upload. All relevant members are hashed above.
print('All three reports and their declared output members verified.')

In [ ]:
# 2 — Validate both manifests and every prediction against record, label, fold and group.
manifest={}
train={}
for stage in (1,2):
 name=f'stage{stage}_candidate_manifest.csv'
 raw=P1.read(name)
 assert sha(raw)==EXPECTED_MANIFEST[name]
 d=table(raw)
 assert len(d)=={1:3268,2:1584}[stage]
 assert d.record_id.is_unique and d.sequence_sha256.is_unique
 assert d.sequence.str.fullmatch('[ACGT]{200}').all()
 assert all(sha(s.encode())==h for s,h in zip(d.sequence,d.sequence_sha256))
 assert d.record_id.eq(d.sequence_sha256.str[:16]).all()
 assert set(d.role)=={'train_candidate','legacy_test','excluded_related_to_legacy_test'}
 t=d[d.role=='train_candidate'].copy()
 assert len(t)=={1:2838,2:1376}[stage]
 assert set(t.fold_5)==set(map(str,range(5)))
 assert t.groupby('similarity_group_id').fold_5.nunique().max()==1
 assert not set(t.similarity_group_id)&set(d.loc[d.role=='legacy_test','similarity_group_id'])
 manifest[stage],train[stage]=d,t
 print(f'Stage {stage}:', d.role.value_counts().to_dict(), 'train labels',t.label.value_counts().to_dict())
shared=train[2].set_index('record_id').similarity_group_id.to_dict()
assert set(shared)<=set(train[1].record_id)
assert all(train[1].set_index('record_id').loc[rid,'similarity_group_id']==g for rid,g in shared.items())

pred=table(P2.read('all_oof_predictions.csv'))
fold=table(P2.read('fold_metrics.csv'))
comparison=table(P2.read('comparison_mean_fold_vs_pooled.csv'))
robust=table(P3.read('phase3_model_robustness.csv'))
paired=table(P3.read('phase3_paired_bootstrap_intervals.csv'))
coverage=table(P3.read('phase3_matching_coverage.csv'))
shift=table(P3.read('phase3_fold_score_shift.csv'))
assert len(pred)==37926 and pred.model.nunique()==9
assert not pred.duplicated(['stage','model','record_id']).any()
assert pred.config_hash.eq(EXPECTED_CONFIG_HASH).all()
pred['score']=pd.to_numeric(pred.score,errors='raise')
assert np.isfinite(pred.score).all() and pred.score.between(0,1).all()
for stage in (1,2):
 truth=train[stage].set_index('record_id')
 for model, part in pred[pred.stage==str(stage)].groupby('model'):
  assert len(part)==len(truth) and set(part.record_id)==set(truth.index)
  expected=truth.loc[part.record_id]
  for col in ('label','fold_5','similarity_group_id'):
   assert part[col].tolist()==expected[col].tolist(), (stage,model,col)
 assert not pred[(pred.stage==str(stage))&(pred.fold_5=='')].shape[0]
assert len(fold)==90 and len(comparison)==18 and len(robust)==18
assert set(comparison.model)==set(pred.model)==set(robust.model)
# Recalculate a reference fold's AUROC from individual scores.
sample=pred[(pred.stage=='1')&(pred.model=='finetuned_dnabert2')&(pred.fold_5=='0')]
reported=fold[(fold.stage=='1')&(fold.model=='finetuned_dnabert2')&(fold.fold_5=='0')]
assert abs(roc_auc_score(sample.label.astype(int),sample.score)-float(reported.roc_auc.iloc[0]))<1e-9
print('Verified all',len(pred),'OOF rows, nine arms, both stages, and grouped fold assignments.')

In [ ]:
# 3 — Build manuscript tables with stage-appropriate, mean-of-fold comparisons.
numeric_cols=['mean_fold_auroc','sd_fold_auroc','mean_fold_auprc','pooled_oof_auroc','pooled_oof_auprc']
for col in numeric_cols: comparison[col]=pd.to_numeric(comparison[col])
for col in ('mean_fold_auroc','pooled_oof_auroc','same_gc_count_mean_fold_auroc',
            'matched_pairs_total','matched_records_total_across_folds','records_total'):
 robust[col]=pd.to_numeric(robust[col])
comparison['stage']=comparison.stage.astype(int); robust['stage']=robust.stage.astype(int)
main=comparison.merge(robust[['stage','model','same_gc_count_mean_fold_auroc','matched_pairs_total',
 'matched_records_total_across_folds','records_total']],on=['stage','model'],validate='one_to_one')
assert np.allclose(main.mean_fold_auroc, robust.set_index(['stage','model']).loc[
 list(zip(main.stage,main.model)),'mean_fold_auroc'].to_numpy())
main['fold_pool_gap']=main.mean_fold_auroc-main.pooled_oof_auroc
main=main.sort_values(['stage','mean_fold_auroc'],ascending=[True,False]).reset_index(drop=True)
main.to_csv(WORK/'table1_main_model_comparison.csv',index=False)

roles=[]
for stage,d in manifest.items():
 t=train[stage]; counts=t.similarity_group_id.value_counts()
 roles.append({'stage':stage,'total_unique_records':len(d),'train_candidate':len(t),
  'excluded_related_to_legacy_test':int((d.role=='excluded_related_to_legacy_test').sum()),
  'legacy_test_secondary':int((d.role=='legacy_test').sum()),
  'train_positive':int((t.label=='1').sum()),'train_negative':int((t.label=='0').sum()),
  'train_similarity_groups':len(counts),'groups_with_multiple_records':int((counts>1).sum()),
  'largest_group_records':int(counts.max())})
audit=pd.DataFrame(roles)
audit.to_csv(WORK/'table2_dataset_audit.csv',index=False)
paired.to_csv(WORK/'table3_exploratory_conditional_bootstrap.csv',index=False)
coverage.to_csv(WORK/'table4_exact_gc_matching_coverage.csv',index=False)
print('MEAN OF FIVE FOLDS (primary descriptive metric):')
print(main[['stage','model','mean_fold_auroc','mean_fold_auprc',
 'same_gc_count_mean_fold_auroc','fold_pool_gap']].round(4).to_string(index=False))
print('\nDataset audit:'); print(audit.to_string(index=False))

In [ ]:
# 4 — GC-stratified OOF error and score diagnostics, without new model fitting.
# Fixed threshold 0.5 is reported descriptively; do not tune it on held-out folds.
frames=[]
for stage in (1,2):
 t=train[stage][['record_id','sequence','label','fold_5','similarity_group_id']].copy()
 t['gc_count']=t.sequence.str.count('G')+t.sequence.str.count('C')
 t['gc_fraction']=t.gc_count/200
 t['group_size']=t.similarity_group_id.map(t.similarity_group_id.value_counts())
 p=pred[pred.stage==str(stage)].drop(columns=['label','fold_5','similarity_group_id'])
 joined=p.merge(t,on='record_id',validate='many_to_one')
 assert len(joined)==len(t)*9
 frames.append(joined)
joined=pd.concat(frames,ignore_index=True)
joined['label']=joined.label.astype(int)
joined['pred_label']=(joined.score>=.5).astype(int)
joined['error']=(joined.pred_label!=joined.label).astype(int)
joined['brier_row']=(joined.score-joined.label)**2
# Fixed bins across both stages preserve comparability and avoid quantile edge instability.
joined['gc_bin']=pd.cut(joined.gc_fraction,[0,.35,.45,.55,.65,1.0],include_lowest=True,
                         labels=['≤35%','35–45%','45–55%','55–65%','>65%'])
rows=[]
for (stage,model,gc_bin),g in joined.groupby(['stage','model','gc_bin'],observed=True):
 if len(g)==0:continue
 rows.append({'stage':stage,'model':model,'gc_bin':str(gc_bin),'n':len(g),
  'positive_fraction':g.label.mean(),'error_rate_at_0_5':g.error.mean(),
  'brier':g.brier_row.mean(),
  'auroc':roc_auc_score(g.label,g.score) if g.label.nunique()==2 else np.nan})
gcdiag=pd.DataFrame(rows)
gcdiag.to_csv(WORK/'table5_gc_stratified_oof_diagnostics.csv',index=False)
# Group-size descriptors. A large sequence-similarity group can still be held in one fold.
groupdiag=joined.groupby(['stage','model', 'group_size'],as_index=False).agg(
 n=('record_id','size'),positive_fraction=('label','mean'),error_rate_at_0_5=('error','mean'),
 mean_brier=('brier_row','mean'))
groupdiag.to_csv(WORK/'table6_similarity_group_size_diagnostics.csv',index=False)
print('Saved GC and similarity-group descriptive diagnostics; counts and rates only, no causal claims.')

In [ ]:
# 5 — Manuscript figures (PNG and PDF). The Phase 3 exact-GC endpoint is already computed.
plt.rcParams.update({'figure.dpi':130,'font.size':9,'axes.spines.top':False,'axes.spines.right':False})
colors={'gc_logreg':'#999999','k3_logreg':'#7c9c65','spaced6_4_logreg':'#238b45',
        'small_cnn':'#8c6bb1','frozen_dnabert_legacy':'#4679a8',
        'frozen_dnabert2':'#1f78b4','frozen_gena_lm':'#338fb3',
        'finetuned_dnabert_legacy':'#db6c28','finetuned_dnabert2':'#bd3c2f'}

def savefig(name):
 for ext in ('png','pdf'): plt.savefig(WORK/f'{name}.{ext}',bbox_inches='tight')
 plt.show(); plt.close()

fig,axs=plt.subplots(1,2,figsize=(12,5),sharex=True)
for ax,stage in zip(axs,(1,2)):
 v=main[main.stage==stage].sort_values('mean_fold_auroc')
 yy=np.arange(len(v)); ax.barh(yy,v.mean_fold_auroc,color=[colors.get(x,'#777') for x in v.model])
 ax.set_yticks(yy,v.model); ax.set_xlim(.48,.95)
 ax.set_title(f'Stage {stage} — mean five-fold AUROC')
 ax.set_xlabel('Mean per-fold AUROC (grouped OOF)')
 ax.axvline(.5,color='black',lw=.6,ls=':')
 for i,val in enumerate(v.mean_fold_auroc):ax.text(val+.005,i,f'{val:.3f}',va='center',fontsize=8)
fig.suptitle('Main benchmark: descriptive five-fold ranking')
fig.tight_layout();savefig('figure1_grouped_cv_mean_fold_auroc')

fig,axs=plt.subplots(1,2,figsize=(12,5),sharex=True)
for ax,stage in zip(axs,(1,2)):
 v=main[main.stage==stage].sort_values('mean_fold_auroc')
 yy=np.arange(len(v))
 ax.scatter(v.mean_fold_auroc,yy,label='Raw',marker='o',color='#324b7c')
 ax.scatter(v.same_gc_count_mean_fold_auroc,yy,label='Same GC count',marker='s',color='#c25427')
 for i,row in enumerate(v.itertuples()):
  ax.plot([row.same_gc_count_mean_fold_auroc,row.mean_fold_auroc],[i,i],color='#aaa',zorder=-1)
 ax.set_yticks(yy,v.model);ax.set_xlim(.45,.95);ax.axvline(.5,color='black',lw=.6,ls=':')
 ax.set_title(f'Stage {stage}');ax.set_xlabel('Mean per-fold AUROC')
 axs[0].legend(loc='lower right',fontsize=8)
fig.suptitle('Sequence-composition sensitivity: pairs with identical GC count')
fig.tight_layout();savefig('figure2_raw_vs_exact_gc_auroc')

fig,axs=plt.subplots(1,2,figsize=(11,4.2))
for ax,stage in zip(axs,(1,2)):
 v=main[main.stage==stage].sort_values('fold_pool_gap',ascending=False)
 ax.bar(np.arange(len(v)),v.fold_pool_gap,color=[colors.get(x,'#777') for x in v.model])
 ax.set_xticks(np.arange(len(v)),v.model,rotation=70,ha='right')
 ax.axhline(0,color='black',lw=.7);ax.set_title(f'Stage {stage}')
 ax.set_ylabel('Mean-fold minus pooled OOF AUROC')
fig.suptitle('Fold-score shifts make pooled and mean-of-fold AUROC differ')
fig.tight_layout();savefig('figure3_fold_vs_pooled_gap')

fig,axs=plt.subplots(1,2,figsize=(11,4),sharey=True)
for ax,stage in zip(axs,(1,2)):
 d=audit[audit.stage==stage].iloc[0]
 names=['Training candidates','Excluded near legacy test','Historical test']
 vals=[d.train_candidate,d.excluded_related_to_legacy_test,d.legacy_test_secondary]
 bars=ax.bar(names,vals,color=['#2a9d8f','#d18b47','#87919e'])
 ax.bar_label(bars,padding=3);ax.set_title(f'Stage {stage}')
 ax.tick_params(axis='x',labelrotation=14)
 axs[0].set_ylabel('Unique sequences')
fig.suptitle('Dataset roles after sequence-similarity audit')
fig.tight_layout();savefig('figure4_manifest_roles')
print('Saved four figures in both PNG and PDF.')

In [ ]:
# 6 — Write a compact synthesis and provenance report; package all manuscript outputs.
def pick(stage,model):
 r=main[(main.stage==stage)&(main.model==model)].iloc[0]
 return {'mean_fold_auroc':float(r.mean_fold_auroc),'mean_fold_auprc':float(r.mean_fold_auprc),
         'same_gc_count_mean_fold_auroc':float(r.same_gc_count_mean_fold_auroc),
         'pooled_oof_auroc':float(r.pooled_oof_auroc)}
key={str(stage):{model:pick(stage,model) for model in ('gc_logreg','spaced6_4_logreg',
         'finetuned_dnabert_legacy','finetuned_dnabert2')} for stage in (1,2)}
report={'title':'Phase 8 main benchmark synthesis','sources':{
 'phase1_report_sha256':EXPECTED_REPORT_SHA['audit_run_summary.json'],
 'phase2_1_report_sha256':EXPECTED_REPORT_SHA['correction_report.json'],
 'phase3_report_sha256':EXPECTED_REPORT_SHA['phase3_run_report.json'],
 'phase1_manifest_sha256':EXPECTED_MANIFEST,'phase2_config_hash':EXPECTED_CONFIG_HASH},
 'evaluation':'Sequence-similarity-grouped five-fold OOF; primary descriptive ranking uses arithmetic mean of fold AUROC.',
 'key_results':key,'limitations':r1['limitations']+r2['limitations']+r3['limitations']+[
  'This notebook summarizes existing model predictions; no new model training or external validation.',
  'Stage 1 and Stage 2 share sequences/groups; treat them as related tasks, not independent cohorts.',
  'GC strata and classification errors use the prespecified threshold 0.5 descriptively.',
  'Conditional Phase 3 bootstrap contrasts were chosen after reviewing Phase 2, with no multiplicity adjustment.']}
(WORK/'phase8_synthesis_report.json').write_text(json.dumps(report,indent=2,sort_keys=True)+'\n')
readme="""# Phase 8 main benchmark synthesis

Inputs: Phase 1 audit, Phase 2.1 corrected OOF results, Phase 3 robustness archive.

- table1: descriptive mean-of-fold and pooled ranking plus GC matched AUROC.
- table2: audited role counts and fold-group descriptors.
- table3: Phase 3 exploratory conditional bootstrap intervals copied with original source semantics.
- table4: exact-GC pair matching coverage.
- tables5–6: new descriptive OOF GC/error/group-size summaries.
- figures1–4: corresponding visual summaries, PNG and PDF.

No Phase 4–7 data or model checkpoints are included. The legacy test has been examined previously. No independent-test claim or biological motif mechanism is supported here.
"""
(WORK/'README.md').write_text(readme)
files=sorted(p for p in WORK.iterdir() if p.is_file())
report['output_sha256']={p.name:sha(p.read_bytes()) for p in files if p.name!='phase8_synthesis_report.json'}
(WORK/'phase8_synthesis_report.json').write_text(json.dumps(report,indent=2,sort_keys=True)+'\n')
ZIP=WORK.parent/'enhancer_phase8_main_synthesis.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for file in sorted(WORK.iterdir()):
  if file.is_file():z.write(file,arcname=file.name)
with zipfile.ZipFile(ZIP) as z:
 assert z.testzip() is None
 assert {x.filename for x in z.infolist()}=={p.name for p in WORK.iterdir() if p.is_file()}
print('Download:', ZIP)
print('ZIP size MB:',round(ZIP.stat().st_size/1e6,2))
print('Stage 1 DNABERT2:', key['1']['finetuned_dnabert2'])
print('Stage 2 DNABERT2:', key['2']['finetuned_dnabert2'])

## How to use the output

Use `table1_main_model_comparison.csv` for the primary grouped-CV result, `table2_dataset_audit.csv` for flow counts, and Figure 2 plus `table3_exploratory_conditional_bootstrap.csv` for composition sensitivity. The latter intervals are exploratory and conditional on already trained models. Do **not** describe the old independent split as untouched. Keep the chromosome-heldout MPRA results (Phases 6–7) as a separate external extension in the manuscript.